# 04 - Customer Feature Engineering (RFM)

Goal: turn transaction-level data into one row per customer, with features that
describe their purchasing behavior. These features feed segmentation (05),
churn prediction (07), and the recommender (08).

We use `retail_customer_level.csv` — the version with missing Customer IDs
already dropped, since every feature here is customer-scoped.

**Cross-version compatibility fix:** this environment's pandas (3.0) defaults to the new `StringDtype` for text columns (`future.infer_string=True`). Anything fitted or computed here (OneHotEncoder categories, product-name lookups) would then get pickled with that dtype -- which older/different pandas installs (e.g. on the machine actually running the dashboard) cannot unpickle, raising `NotImplementedError` on load. Forcing `pd.set_option('future.infer_string', False)` up front keeps everything on plain `object` dtype, which pickles compatibly across pandas versions.


## Recency, Frequency, Monetary

Both Frequency and Monetary are extremely right-skewed:

- Monetary: median ~£868, max ~£580,987 (a ~670x spread)
- Frequency: median 3 orders, max 398 orders

If we feed raw values into KMeans later, distance calculations will be dominated
by a handful of wholesale-scale outliers, and the clustering will basically just
separate "whales" from "everyone else" instead of finding meaningful behavioral
groups. We add log-transformed versions below to use for clustering, while
keeping the raw values for reporting and business interpretation.

## Additional behavioral features

`AvgOrderValue` is the average line value, not average order total — a proxy for
whether a customer tends to buy cheap or expensive items, not how large their
baskets are. `UniqueProducts` measures range of interest. `PurchaseDays` counts
distinct calendar days with a purchase (different from Frequency/invoice count,
since a customer could place multiple invoices in one day).

## Country

13 customers have transactions logged under more than one country. Rather than
dropping them or picking arbitrarily, we assign each customer their single most
frequent country.

We use a binary `IsUK` flag rather than one-hot encoding all 43 countries.
From the EDA notebook, UK accounts for ~85% of revenue and the vast majority of
customers — every other country individually has too few customers to be a
statistically meaningful category on its own. `IsUK` captures the one split
that actually has enough data behind it; the raw `Country` column is kept for
reference/filtering, not as a model feature.

## Merge into one customer feature table

## Save

In [1]:
import pandas as pd
pd.set_option('future.infer_string', False)
import pandas as pd
import numpy as np

customer_df = pd.read_csv("../data/processed/retail_customer_level.csv", parse_dates=["InvoiceDate"], low_memory=False)
print("shape:", customer_df.shape)

reference_date = customer_df["InvoiceDate"].max() + pd.Timedelta(days=1)
print("reference_date:", reference_date)

rfm = customer_df.groupby("Customer ID").agg(
    Recency=("InvoiceDate", lambda x: (reference_date - x.max()).days),
    Frequency=("Invoice", "nunique"),
    Monetary=("TotalAmount", "sum")
).reset_index()
print("rfm shape:", rfm.shape)
print(rfm.describe())

rfm["Frequency_log"] = np.log1p(rfm["Frequency"])
rfm["Monetary_log"] = np.log1p(rfm["Monetary"])

extra_features = customer_df.groupby("Customer ID").agg(
    AvgOrderValue=("TotalAmount", "mean"),
    UniqueProducts=("StockCode", "nunique"),
    PurchaseDays=("InvoiceDate", lambda x: x.dt.date.nunique())
).reset_index()

country_per_customer = customer_df.groupby("Customer ID")["Country"].nunique()
print("multi-country customers:", (country_per_customer > 1).sum())

customer_country = (
    customer_df.groupby("Customer ID")["Country"]
    .agg(lambda x: x.mode().iloc[0])
    .reset_index()
)
customer_country["IsUK"] = (customer_country["Country"] == "United Kingdom").astype(int)

customer_features = (
    rfm
    .merge(extra_features, on="Customer ID")
    .merge(customer_country, on="Customer ID")
)
print("customer_features shape:", customer_features.shape)
print("nulls:\n", customer_features.isnull().sum())

customer_features.to_csv("../data/processed/customer_features.csv", index=False)
print("saved:", customer_features.shape)


shape: (779425, 9)
reference_date: 2011-12-10 12:50:00
rfm shape: (5878, 4)
        Customer ID      Recency    Frequency       Monetary
count   5878.000000  5878.000000  5878.000000    5878.000000
mean   15315.313542   201.331916     6.289384    2955.904095
std     1715.572666   209.338707    13.009406   14440.852688
min    12346.000000     1.000000     1.000000       2.950000
25%    13833.250000    26.000000     1.000000     342.280000
50%    15314.500000    96.000000     3.000000     867.740000
75%    16797.750000   380.000000     7.000000    2248.305000
max    18287.000000   739.000000   398.000000  580987.040000
multi-country customers: 13
customer_features shape: (5878, 11)
nulls:
 Customer ID       0
Recency           0
Frequency         0
Monetary          0
Frequency_log     0
Monetary_log      0
AvgOrderValue     0
UniqueProducts    0
PurchaseDays      0
Country           0
IsUK              0
dtype: int64
saved: (5878, 11)
